# E6 机械臂 VLA 云端微调与多检查点物理优选（Google Colab 完备版）

针对 **600 条** E6 双视角高质量示教数据（4 类任务各 150 条，共 **36,000 帧**，`eye` 全局 + 避障 `wrist` 腕部近景，`10 Hz`，含 `0.85~1.15` 灯光随机化），适配 `lerobot 0.7.0+`，内置 **防断网双份日志 `train.log`** 与 **省计算单元（CU）保护**：

1. **路线 A2：`smolvla_s2`（当前默认，二阶段热启动救活 SmolVLA）**：
   - 从第一阶段 Drive 冠军权重（`outputs/smolvla_e6/best_pretrained_model`）热启动，**解冻 16 层 VLM 视觉-语言融合主干 + 动作专家（共 `306.5M` 可训练参数）**，仅锁住底层 SigLIP 视觉编码器；
   - **选 `A100 (40GB)`**：`batch_size=64`、`steps=12,000`、`lr=5e-5`（500 步预热 + 12,000 步余弦衰减），每 `3,000` 步存盘（共 4 个检查点全进 MuJoCo 实测挑冠军），约 **60～80 分钟**，消耗约 **`13～18 CU`**。
2. **路线 A1：`smolvla`（从 `lerobot/smolvla_base` 从头微调基线）**。
3. **路线 B：`pi05`（`lerobot/pi05_base` + 高秩 LoRA `r=32` 进阶对比）**：约 `35 亿`参数，A100 `steps=6,000`，约 **`20～26 CU`**（需先在 HuggingFace 同意 `google/paligemma-3b-pt-224` 协议并准备 Read Token）。

> **出发前准备（Mac 已 0 CU 转好缓存时只需 1 步）**：
> 1. 将 Mac 上生成好的 `ml/m6_vla/data/lerobot_e6_pick_v600.tar` 上传到 Google Drive：`MyDrive/robot-navigation-lab/lerobot_e6_pick_v600.tar`（确认目录里同时有之前传的 `e6_colab_bundle.zip` 供提取 `e6_assets`）。
> 2. 运行时选择 **A100 GPU**，点菜单 **代码执行程序 → 全部运行**。
> 3. **断网也不慌**：第 4 格会把训练实时日志同步写入本地 `/content/outputs/<run>/train.log` 和 Google Drive 同目录 `train.log`；重连页面后打开左侧 `>_ 终端` 执行 `tail -f /content/outputs/smolvla_e6_s2/train.log` 即可实时查看最新步数与 `loss`！


In [ ]:
# 第 1 格：检查 GPU、挂载 Google Drive、拉取最新仓库并准备 E6 仿真资产
!nvidia-smi || echo "ℹ️ 当前为免费 CPU 运行时（可先跑第 1~3 格缓存数据集，0 CU）"

import shutil
import subprocess
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_BASE = Path("/content/drive/MyDrive/robot-navigation-lab")
DATASET_CACHE_NAME = "lerobot_e6_pick_v600.tar"
CACHE_TAR = DRIVE_BASE / DATASET_CACHE_NAME
BUNDLE_ZIP = DRIVE_BASE / "e6_colab_bundle.zip"
assert CACHE_TAR.is_file() or BUNDLE_ZIP.is_file(), (
    f"未在 {DRIVE_BASE} 找到 {DATASET_CACHE_NAME} 或 e6_colab_bundle.zip，请先上传！"
)

REPO_DIR = Path("/content/robot-navigation-lab")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/tdgcdtd/robot-navigation-lab.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

WORK_DIR = Path("/content/work")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

if BUNDLE_ZIP.is_file():
    with zipfile.ZipFile(BUNDLE_ZIP, "r") as zf:
        if CACHE_TAR.is_file():
            # 命中 .tar 缓存时仅提取轻量的 e6_assets（跳过旧视频解压，节省时间）
            asset_members = [m for m in zf.namelist() if m.startswith("e6_assets/")]
            for m in asset_members:
                zf.extract(m, WORK_DIR)
        else:
            zf.extractall(WORK_DIR)

ASSETS_DST = REPO_DIR / "ml" / "m5_robot" / "assets"
ASSETS_DST.mkdir(parents=True, exist_ok=True)
if (WORK_DIR / "e6_assets").is_dir():
    shutil.copytree(WORK_DIR / "e6_assets", ASSETS_DST, dirs_exist_ok=True)
assert (ASSETS_DST / "e6.xml").is_file(), f"缺少 E6 模型文件：{ASSETS_DST / 'e6.xml'}"

RAW_DATA_DIR = WORK_DIR / "e6_pick"
if not CACHE_TAR.is_file():
    assert (RAW_DATA_DIR / "meta.json").is_file(), "未命中 .tar 缓存且解压包缺少 e6_pick/meta.json"
print("✅ 环境、仓库与 E6 资产就绪 | 数据缓存命中:", CACHE_TAR.is_file())


In [ ]:
# 第 2 格：选择训练路线与安装依赖（显式安装 lerobot[dataset,training,...] 并立即验通导入）
import importlib.util
import shutil
import subprocess
import sys

MODEL_CHOICE = "smolvla_s2"  # "smolvla_s2"（默认：二阶段解冻 VLM）| "smolvla"（从头微调）| "pi05"（A100 LoRA）
AUTO_DISCONNECT = True       # True = 跑完第 6 格后等 15 秒自动释放 GPU，避免白扣计算单元
DATASET_CACHE_NAME = "lerobot_e6_pick_v600.tar"

assert MODEL_CHOICE in ("smolvla_s2", "smolvla", "pi05"), f"不支持的 MODEL_CHOICE: {MODEL_CHOICE}"

extra = "pi,peft" if MODEL_CHOICE == "pi05" else "smolvla"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", f"lerobot[dataset,training,{extra}]", "mujoco", "imageio", "imageio-ffmpeg"],
    check=True,
)

if MODEL_CHOICE == "pi05":
    from huggingface_hub import login
    print("请在弹窗中输入你的 Hugging Face Read Token（需已在 HF 网站同意 google/paligemma-3b-pt-224 协议）：")
    login()

!which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)

import lerobot
print("✅ lerobot 版本:", getattr(lerobot, "__version__", "unknown"))

LeRobotDataset = None
for mod in ("lerobot.datasets", "lerobot.datasets.lerobot_dataset", "lerobot.common.datasets.lerobot_dataset"):
    try:
        LeRobotDataset = getattr(__import__(mod, fromlist=["LeRobotDataset"]), "LeRobotDataset")
        break
    except (ImportError, AttributeError):
        continue
assert LeRobotDataset is not None, "❌ LeRobotDataset 导入失败，请检查 pip 安装输出！"

has_train = bool(shutil.which("lerobot-train")) or any(
    importlib.util.find_spec(m) for m in ("lerobot.scripts.lerobot_train", "lerobot.scripts.train")
)
assert has_train, "❌ 未找到 lerobot-train 训练入口，请检查 lerobot[training] 安装！"
print(f"✅ 依赖就绪（当前路线：{MODEL_CHOICE}，自动关机：{AUTO_DISCONNECT}）")


In [ ]:
# 第 3 格：优先 5 秒复用 Drive 预转换缓存；否则流式 H.264 转换并自动缓存到 Drive
import json
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

def run_stream(cmd, env=None):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"命令退出码 {rc}: {' '.join(cmd)}")

LEROBOT_ROOT = Path("/content/lerobot_e6_pick")
SCRIPT_DIR = Path("/content/robot-navigation-lab/ml/m6_vla")
DRIVE_CACHE = Path("/content/drive/MyDrive/robot-navigation-lab") / DATASET_CACHE_NAME

if DRIVE_CACHE.is_file():
    t0 = time.monotonic()
    if LEROBOT_ROOT.exists():
        shutil.rmtree(LEROBOT_ROOT)
    with tarfile.open(DRIVE_CACHE, "r") as tf:
        # 过滤 macOS tar 自带的 AppleDouble 隐藏文件 (._*) 与 .DS_Store，防止 Linux 下 pyarrow 误读损坏的 ._*.parquet
        clean_members = [
            m for m in tf.getmembers()
            if not Path(m.name).name.startswith("._") and Path(m.name).name != ".DS_Store"
        ]
        tf.extractall(LEROBOT_ROOT.parent, members=clean_members)
    for junk in list(LEROBOT_ROOT.rglob("._*")) + list(LEROBOT_ROOT.rglob(".DS_Store")):
        if junk.is_file():
            junk.unlink()
    assert (LEROBOT_ROOT / "meta" / "info.json").is_file(), f"解压缓存后缺少 {LEROBOT_ROOT / 'meta' / 'info.json'}"
    print(f"⚡ 命中 Google Drive 缓存（{DRIVE_CACHE.name}），{time.monotonic() - t0:.1f}s 解压完成，跳过转码！")
else:
    assert Path("/content/work/e6_pick/meta.json").is_file(), "未命中 .tar 缓存且 /content/work/e6_pick 不存在！"
    run_stream([
        sys.executable, str(SCRIPT_DIR / "convert_to_lerobot.py"),
        "--src", "/content/work/e6_pick",
        "--dst", str(LEROBOT_ROOT),
        "--repo-id", "local/e6_pick",
        "--vcodec", "h264",
        "--overwrite",
    ])
    tmp_tar = Path(f"/content/{DATASET_CACHE_NAME}.tmp")
    with tarfile.open(tmp_tar, "w") as tf:
        tf.add(LEROBOT_ROOT, arcname=LEROBOT_ROOT.name)
    DRIVE_CACHE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(tmp_tar, DRIVE_CACHE)
    tmp_tar.unlink(missing_ok=True)
    print(f"💾 已缓存至 Google Drive（{DRIVE_CACHE.stat().st_size / 1024 / 1024:.1f} MB）：{DRIVE_CACHE}")

info = json.loads((LEROBOT_ROOT / "meta" / "info.json").read_text())
stats = json.loads((LEROBOT_ROOT / "meta" / "stats.json").read_text())
print("✅ LeRobot info:", {k: info.get(k) for k in ("total_episodes", "total_frames", "fps", "robot_type")})
print("✅ stats keys for action:", list(stats.get("action", {}).keys()))

LeRobotDataset = None
for mod in ("lerobot.datasets", "lerobot.datasets.lerobot_dataset", "lerobot.common.datasets.lerobot_dataset"):
    try:
        LeRobotDataset = getattr(__import__(mod, fromlist=["LeRobotDataset"]), "LeRobotDataset")
        break
    except (ImportError, AttributeError):
        continue
assert LeRobotDataset is not None, "未能导入 LeRobotDataset"
ds = LeRobotDataset("local/e6_pick", root=LEROBOT_ROOT)
print("✅ 本地 LeRobotDataset 回读验证通过:", len(ds), "帧 | keys:", list(ds[0].keys()))


In [ ]:
# 第 4 格：云端微调（支持二阶段热启动解冻 VLM + 双份防断网 train.log + finally 自动备份）
import importlib.util
import json
import os
import shlex
import shutil
import subprocess
import sys
import time
from pathlib import Path
import torch
from huggingface_hub import hf_hub_download

def run_stream(cmd, env=None, log_path=None, drive_log_path=None):
    log_f = open(log_path, "a", encoding="utf-8") if log_path else None
    last_sync = time.monotonic()
    try:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        for line in p.stdout:
            print(line, end="", flush=True)
            if log_f is not None:
                log_f.write(line)
                log_f.flush()
                if drive_log_path and (time.monotonic() - last_sync) >= 10.0:
                    try:
                        shutil.copy2(log_path, drive_log_path)
                    except OSError:
                        pass
                    last_sync = time.monotonic()
        rc = p.wait()
        if rc != 0:
            raise RuntimeError(f"命令退出码 {rc}: {' '.join(cmd)}")
    finally:
        if log_f is not None:
            log_f.flush()
            log_f.close()
            if drive_log_path and Path(log_path).is_file():
                try:
                    shutil.copy2(log_path, drive_log_path)
                except OSError:
                    pass

LEROBOT_ROOT = Path("/content/lerobot_e6_pick")
STAGE1_CKPT = Path("/content/drive/MyDrive/robot-navigation-lab/outputs/smolvla_e6/best_pretrained_model")

if MODEL_CHOICE == "smolvla_s2":
    assert (STAGE1_CKPT / "config.json").is_file(), (
        f"未找到第一阶段检查点：{STAGE1_CKPT}，请确认 Drive 路径完整！"
    )
    BASE_MODEL = str(STAGE1_CKPT)
    RUN_NAME = "smolvla_e6_s2"
    POLICY_TYPE_HELP = "smolvla"
elif MODEL_CHOICE == "smolvla":
    BASE_MODEL = "lerobot/smolvla_base"
    RUN_NAME = "smolvla_e6"
    POLICY_TYPE_HELP = "smolvla"
else:
    BASE_MODEL = "lerobot/pi05_base"
    RUN_NAME = "pi05_lora_e6"
    POLICY_TYPE_HELP = "pi05"

OUT_DIR = Path(f"/content/outputs/{RUN_NAME}")
DRIVE_OUT = Path("/content/drive/MyDrive/robot-navigation-lab/outputs") / RUN_NAME
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
TRAIN_LOG = OUT_DIR / "train.log"
DRIVE_TRAIN_LOG = DRIVE_OUT / "train.log"

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0.0
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"🖥️ 当前设备: {gpu_name} ({vram_gb:.1f} GB VRAM) | 输出目录: {OUT_DIR}")

if shutil.which("lerobot-train"):
    train_bin = ["lerobot-train"]
else:
    for mod in ("lerobot.scripts.lerobot_train", "lerobot.scripts.train"):
        if importlib.util.find_spec(mod):
            train_bin = [sys.executable, "-m", mod]
            break
    else:
        raise RuntimeError("未找到 lerobot-train 命令或训练入口模块")

help_proc = subprocess.run(train_bin + [f"--policy.type={POLICY_TYPE_HELP}", "--help"], capture_output=True, text=True)
help_text = (help_proc.stdout or "") + "\n" + (help_proc.stderr or "")

# 支持本地检查点与 HF Hub 仓库自动解析图像输入键名（过滤掉历史占位键 empty_camera_*）
local_cfg = Path(BASE_MODEL) / "config.json"
cfg_path = local_cfg if local_cfg.is_file() else Path(hf_hub_download(BASE_MODEL, "config.json"))
base_cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
vis_keys = [
    k for k, v in base_cfg.get("input_features", {}).items()
    if ((isinstance(v, dict) and v.get("type") == "VISUAL") or "observation.images." in k)
    and "empty_camera_" not in k
]
assert vis_keys, f"未能从 {cfg_path} 解析出图像输入键名！"

eye_dst = next((k for k in vis_keys if "wrist" not in k.lower()), vis_keys[0])
wrist_dst = next(
    (k for k in vis_keys if k != eye_dst and any(tag in k.lower() for tag in ("wrist", "camera2", "image2"))),
    vis_keys[1] if len(vis_keys) > 1 else None,
)
rename_map = {"observation.images.eye": eye_dst}
if wrist_dst:
    rename_map["observation.images.wrist"] = wrist_dst
empty_cameras = max(0, len(vis_keys) - len(rename_map))

print("底座期望真实相机:", vis_keys)
print("自动构造 rename_map:", rename_map, "| empty_cameras:", empty_cameras)

if MODEL_CHOICE == "smolvla_s2":
    batch_size = 64 if vram_gb >= 35 else 32
    if vram_gb < 35:
        print("⚠️ 检测到当前显卡显存 < 35GB（大概率被 Colab 自动降级成了 L4），将退回 batch_size=32，速度比 A100 慢 ~2.5 倍；建议先断开并重新申请 A100，以免耗时过长！", flush=True)
    cmd = train_bin + [
        f"--policy.path={BASE_MODEL}",
        "--dataset.repo_id=local/e6_pick",
        f"--dataset.root={LEROBOT_ROOT}",
        f"--output_dir={OUT_DIR}",
        f"--job_name={RUN_NAME}",
        "--policy.train_expert_only=false",
        "--policy.freeze_vision_encoder=true",
        "--policy.optimizer_lr=5e-5",
        "--policy.scheduler_warmup_steps=500",
        "--policy.scheduler_decay_steps=12000",
        "--policy.scheduler_decay_lr=2.5e-6",
        f"--policy.empty_cameras={empty_cameras}",
        f"--rename_map={json.dumps(rename_map)}",
        f"--batch_size={batch_size}",
        "--steps=12000",
        "--save_freq=3000",
        "--log_freq=200",
    ]
elif MODEL_CHOICE == "smolvla":
    batch_size = 64 if vram_gb >= 35 else 32
    cmd = train_bin + [
        f"--policy.path={BASE_MODEL}",
        "--dataset.repo_id=local/e6_pick",
        f"--dataset.root={LEROBOT_ROOT}",
        f"--output_dir={OUT_DIR}",
        f"--job_name={RUN_NAME}",
        f"--policy.empty_cameras={empty_cameras}",
        f"--rename_map={json.dumps(rename_map)}",
        f"--batch_size={batch_size}",
        "--steps=20000",
        "--save_freq=5000",
        "--log_freq=200",
    ]
else:
    batch_size = 32 if vram_gb >= 70 else 16
    cmd = train_bin + [
        "--policy.type=pi05",
        f"--policy.pretrained_path={BASE_MODEL}",
        "--dataset.repo_id=local/e6_pick",
        f"--dataset.root={LEROBOT_ROOT}",
        f"--output_dir={OUT_DIR}",
        f"--job_name={RUN_NAME}",
        "--peft.method_type=LORA",
        "--peft.r=32",
        "--peft.lora_alpha=32",
        "--policy.dtype=bfloat16",
        "--policy.gradient_checkpointing=true",
        f"--policy.empty_cameras={empty_cameras}",
        f"--rename_map={json.dumps(rename_map)}",
        f"--batch_size={batch_size}",
        "--steps=6000",
        "--save_freq=2000",
        "--log_freq=100",
    ]

if "--policy.device" in help_text:
    cmd.append("--policy.device=cuda")
if "--policy.push_to_hub" in help_text:
    cmd.append("--policy.push_to_hub=false")
if "--num_workers" in help_text:
    cmd.append("--num_workers=4")
if "--accelerator.mixed_precision" in help_text:
    cmd.append("--accelerator.mixed_precision=bf16")
elif MODEL_CHOICE in ("smolvla", "smolvla_s2") and "--policy.use_amp" in help_text:
    cmd.append("--policy.use_amp=true")
if "--trackio.enable" in help_text:
    cmd.append("--trackio.enable=false")
elif "--wandb.enable" in help_text:
    cmd.append("--wandb.enable=false")

# lerobot-train 要求 output_dir 尚不存在；先删除空占位目录并确保仅建好父目录
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
LOCAL_LOG_MIRROR = Path(f"/content/{RUN_NAME}_train.log")
LOCAL_LOG_MIRROR.write_text("", encoding="utf-8")

print(f"📝 防断网实时日志：{LOCAL_LOG_MIRROR}（每 10 秒同步一份到 {DRIVE_TRAIN_LOG}）")
print(f"   如果浏览器断网，可在 Colab 终端执行：tail -f {LOCAL_LOG_MIRROR}\n")
print("执行命令：\n" + " ".join(shlex.quote(c) for c in cmd), flush=True)

def _backup_latest_checkpoint():
    ckpt_root = OUT_DIR / "checkpoints"
    if not ckpt_root.is_dir():
        print("⚠️ 尚未产生任何检查点可备份")
        return None
    target = ckpt_root / "last" / "pretrained_model"
    if not target.is_dir():
        subs = sorted(p for p in ckpt_root.iterdir() if (p / "pretrained_model").is_dir())
        if not subs:
            return None
        target = subs[-1] / "pretrained_model"
    drive_ckpt = DRIVE_OUT / "pretrained_model"
    if drive_ckpt.exists():
        shutil.rmtree(drive_ckpt)
    shutil.copytree(target, drive_ckpt)
    # 先在本地 NVMe 极速压缩，再顺序写入 Drive，避免跨挂载小文件卡顿
    tmp_zip = Path(shutil.make_archive(f"/content/{RUN_NAME}_last_pretrained_model", "zip", root_dir=target))
    drive_zip = DRIVE_OUT / tmp_zip.name
    shutil.copy2(tmp_zip, drive_zip)
    tmp_zip.unlink(missing_ok=True)
    if LOCAL_LOG_MIRROR.is_file():
        shutil.copy2(LOCAL_LOG_MIRROR, DRIVE_TRAIN_LOG)
    print(f"✅ 最新检查点已备份至 Google Drive：{drive_ckpt}")
    print(f"📦 最新检查点压缩包：{drive_zip}")
    return drive_ckpt

env_train = os.environ.copy()
env_train["PYTHONUNBUFFERED"] = "1"
SAVED_CKPT_DIR = None
try:
    run_stream(cmd, env=env_train, log_path=LOCAL_LOG_MIRROR, drive_log_path=DRIVE_TRAIN_LOG)
finally:
    SAVED_CKPT_DIR = _backup_latest_checkpoint()


In [ ]:
# 第 5 格：多检查点 MuJoCo 物理实测筛选（全部 4 个检查点 bf16 极速赛马，挑出真冠军备份 Drive）
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

def run_stream(cmd, env=None):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"命令退出码 {rc}: {' '.join(cmd)}")

ckpt_root = OUT_DIR / "checkpoints"
seen_resolved = {}
if ckpt_root.is_dir():
    dirs = sorted(
        (p / "pretrained_model" for p in ckpt_root.iterdir() if (p / "pretrained_model").is_dir()),
        key=lambda p: (p.parent.name == "last", p.parent.name),
    )
    for p in dirs:
        r = p.resolve()
        if r not in seen_resolved:
            seen_resolved[r] = p

candidates = list(seen_resolved.values())[-4:]
if not candidates and SAVED_CKPT_DIR is not None and Path(SAVED_CKPT_DIR).is_dir():
    candidates = [Path(SAVED_CKPT_DIR)]

assert candidates, "未找到任何可用模型检查点，请先完成第 4 格！"
print(f"🔎 将对比评测 {len(candidates)} 个检查点：{[c.parent.name for c in candidates]}")

env_vars = os.environ.copy()
env_vars["MUJOCO_GL"] = "egl"
SCRIPT = "/content/robot-navigation-lab/ml/m6_vla/eval_vla.py"
leaderboard = []

for ckpt_dir in candidates:
    tag = ckpt_dir.parent.name
    vid_path = Path(f"/content/outputs/{RUN_NAME}_eval_{tag}.mp4")
    metrics_path = Path(f"/content/outputs/{RUN_NAME}_metrics_{tag}.json")
    print(f"\n=== 评测检查点 {tag} (20 轮未见随机场景, n_action_steps=5) ===", flush=True)
    run_stream([
        sys.executable, SCRIPT,
        "--checkpoint", str(ckpt_dir),
        "--episodes", "20",
        "--seed", "1000",
        "--light-range", "0.85", "1.15",
        "--n-action-steps", "5",
        "--device", "cuda",
        "--save-video", str(vid_path),
        "--video-episodes", "4",
        "--out-json", str(metrics_path),
    ], env=env_vars)
    m = json.loads(metrics_path.read_text(encoding="utf-8"))
    leaderboard.append({
        "step": tag,
        "ckpt_dir": str(ckpt_dir),
        "video_path": str(vid_path),
        "success": m["success"],
        "episodes": m["episodes"],
        "success_rate": m["success_rate"],
        "mean_err_mm": m["mean_err_mm"],
        "mean_infer_ms": m["mean_infer_ms"],
        "per_task": m["per_task"],
    })

print("\n🏆 多检查点实测排行榜（按成功率优先、平均误差越小越好）：")
ranked = sorted(
    leaderboard,
    key=lambda x: (x["success_rate"], -(x["mean_err_mm"] if x["mean_err_mm"] is not None else 999.0)),
    reverse=True,
)
for rank, row in enumerate(ranked, 1):
    print(f"  #{rank} [{row['step']}] 成功 {row['success']}/{row['episodes']} ({row['success_rate']*100:.1f}%)"
          f" | 平均误差 {row['mean_err_mm']} mm | 推理 {row['mean_infer_ms']} ms")

best = ranked[0]
best_src = Path(best["ckpt_dir"])
best_drive_dir = DRIVE_OUT / "best_pretrained_model"
if best_drive_dir.exists():
    shutil.rmtree(best_drive_dir)
shutil.copytree(best_src, best_drive_dir)

tmp_zip = Path(shutil.make_archive(f"/content/{RUN_NAME}_best_pretrained_model", "zip", root_dir=best_src))
best_zip = DRIVE_OUT / tmp_zip.name
shutil.copy2(tmp_zip, best_zip)
tmp_zip.unlink(missing_ok=True)

best_vid_dst = DRIVE_OUT / f"{RUN_NAME}_best_eval.mp4"
if Path(best["video_path"]).is_file():
    shutil.copy2(Path(best["video_path"]), best_vid_dst)

board_path = DRIVE_OUT / f"{RUN_NAME}_all_ckpts_eval.json"
board_path.write_text(json.dumps(ranked, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\n✅ 冠军检查点 [{best['step']}] 已存入 Google Drive：{best_drive_dir}")
print(f"📦 请下载这包回 Mac 跑 3D 闭环：{best_zip}")
print(f"🎬 冠军三视角录像：{best_vid_dst}")
print(f"📊 完整榜单记录：{board_path}")


In [ ]:
# 第 6 格：安全保护——等待 15 秒后自动释放 GPU，避免白白消耗每月 200 计算单元
import time

if AUTO_DISCONNECT:
    print("⏳ 训练与优选结果已全部存入 Google Drive，15 秒后自动断开并释放 GPU 运行时...")
    time.sleep(15)
    from google.colab import runtime
    runtime.unassign()
else:
    print("ℹ️ AUTO_DISCONNECT=False，GPU 保持开机；用完请务必点击“代码执行程序 -> 断开连接并删除运行时”！")
